<a href="https://colab.research.google.com/github/vichu2k7-crypto/CyberSecurity-Lab/blob/main/U4e2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def handle_outbound_syn(packet, state_table, allow_rules):
    """
    packet: {"src","sport","dst","dport"} outbound SYN.
    If (dst, dport) is permitted by allow_rules, record it in the
    state table so matching return traffic will be permitted.
    """

    key = (packet["dst"], packet["dport"])

    if key in allow_rules:
        state_table[
            (packet["src"], packet["sport"],
             packet["dst"], packet["dport"])
        ] = "ESTABLISHED"

        return "allow"

    return "deny"


def handle_inbound_packet(packet, state_table):
    """
    packet: {"src","sport","dst","dport"} inbound packet.
    Only allowed if it matches an existing state-table entry
    in the reverse direction.
    """

    key = (
        packet["dst"],
        packet["dport"],
        packet["src"],
        packet["sport"]
    )

    if key in state_table:
        return "allow"

    return "deny"

In [3]:
def test_experiment2():

    state_table = {}

    allow_rules = {
        ("93.184.216.34", 443)
    }

    # Outbound connection
    outbound = {
        "src": "10.0.0.20",
        "sport": 51000,
        "dst": "93.184.216.34",
        "dport": 443
    }

    assert handle_outbound_syn(
        outbound,
        state_table,
        allow_rules
    ) == "allow"

    assert (
        outbound["src"],
        outbound["sport"],
        outbound["dst"],
        outbound["dport"]
    ) in state_table


    # Legitimate return traffic
    response = {
        "src": "93.184.216.34",
        "sport": 443,
        "dst": "10.0.0.20",
        "dport": 51000
    }

    assert handle_inbound_packet(
        response,
        state_table
    ) == "allow"


    # Unsolicited inbound traffic
    unsolicited = {
        "src": "203.0.113.9",
        "sport": 4444,
        "dst": "10.0.0.20",
        "dport": 51000
    }

    assert handle_inbound_packet(
        unsolicited,
        state_table
    ) == "deny"


    print("All test cases passed.")


test_experiment2()

All test cases passed.
